# Sanad on Kaggle (free GPU): baseline -> SFT -> DPO, judged by citegate
**Before you run:** right panel -> Settings -> Accelerator: **GPU T4 x2** (or P100) and Internet: **On**. Optional secret `HF_TOKEN` (Add-ons -> Secrets) to publish the adapter.

**Honest status:** the baseline section has been run on a T4; the training sections are being verified. Run top to bottom, stop at the first error and fix the cause before continuing. Every shell step below raises an error if the command fails, so a failed training run cannot silently flow into the evaluation cells. Do not skip the 20-step smoke tests.

In [ ]:
import os
import subprocess
import time
import urllib.request

BASE = "Qwen/Qwen2.5-1.5B-Instruct"  # placeholder: must match configs/sft.toml; check it loads and has a licence you can use
REPO = "https://github.com/ChaiebDhia/sanad.git"
PORT = 8000
URL = f"http://127.0.0.1:{PORT}/v1"


def sh(cmd):
    """Run a shell command, stream its output, and RAISE if it fails (so later cells never run on a failed step)."""
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")


sh("nvidia-smi")

In [ ]:
if not os.path.isdir("sanad"):
    sh(f"git clone {REPO} sanad")
os.chdir("sanad")
sh("git pull")
sh('pip install -q -e ".[train]"')
sh(
    'pip list 2>/dev/null | grep -i -E "^(torch|transformers|peft|trl|bitsandbytes|datasets|accelerate|citegate) "'
)
import torch  # noqa: E402  (imported after the install above)

print(
    "GPU compute capability:",
    torch.cuda.get_device_capability(),
    "(native bf16 needs 8.0 or higher; the scripts choose fp16 below that)",
)

## 1. Data (no GPU needed) and its self-checks

In [ ]:
sh("sanad data-build --out data")
sh("sanad data-check --data data")
sh("python train/sft_qlora.py --dry-run")
sh("python train/dpo.py --dry-run")

## 2. Helper: start/stop the model server (citegate talks to it like any hosted model)

In [ ]:
server = None


def stop():
    global server
    if server:
        server.terminate()
        server.wait()
        server = None
    time.sleep(3)


def start(model, name, adapter=None):
    global server
    stop()
    if adapter and not os.path.exists(os.path.join(adapter, "adapter_config.json")):
        raise FileNotFoundError(
            f"{adapter}/adapter_config.json not found: the training step before this one did not finish"
        )
    cmd = ["sanad", "serve", "--model", model, "--name", name, "--port", str(PORT)]
    if adapter:
        cmd += ["--adapter", adapter]
    server = subprocess.Popen(cmd)
    for _ in range(180):
        if server.poll() is not None:
            raise RuntimeError(f"server exited with code {server.returncode}; read its log above")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
            print("server up")
            return
        except Exception:
            time.sleep(5)
    raise RuntimeError("server did not become healthy in 15 minutes; read its log above")

## 3. Baseline: the untouched base model (record it even if it is bad)
First a 30-case smoke test, then the full citegate golden set (744 cases) and Sanad's dev world.

In [ ]:
start(BASE, "base")
sh(
    f"citegate run --agent llm --provider ollama --base-url {URL} --model base --sample 30 --out runs/smoke-base.jsonl"
)

In [ ]:
sh(
    f"citegate run --agent llm --provider ollama --base-url {URL} --model base --concurrency 8 --out runs/base.jsonl --html runs/base.html --md runs/base.md"
)
sh(f"sanad dev-eval --base-url {URL} --model base --concurrency 8 --out runs/base-dev.jsonl")
sh(f"sanad bench --base-url {URL} --model base --n 100 --concurrency 8 --out runs/base-bench.json")
stop()

## 4. SFT. Smoke test (20 steps) first, then the full run. If the session dies, rerun the full cell with `--resume`.

In [ ]:
sh("python train/sft_qlora.py --max-steps 20")

In [ ]:
sh("python train/sft_qlora.py")
assert os.path.exists("runs/sft/adapter_config.json"), "training finished but no adapter was saved"
print("adapter saved")

In [ ]:
start(BASE, "sft", adapter="runs/sft")
sh(
    f"citegate run --agent llm --provider ollama --base-url {URL} --model sft --concurrency 8 --out runs/sft.jsonl --html runs/sft.html --md runs/sft.md"
)
sh(f"sanad dev-eval --base-url {URL} --model sft --concurrency 8 --out runs/sft-dev.jsonl")
sh(f"sanad bench --base-url {URL} --model sft --n 100 --concurrency 8 --out runs/sft-bench.json")
stop()
sh("citegate compare runs/base.jsonl runs/sft.jsonl --md runs/compare-sft.md")

## 5. DPO on failure-mode pairs (starts from the merged SFT model)

In [ ]:
sh(f"python train/merge_adapter.py --base {BASE} --adapter runs/sft --out runs/sft-merged")
assert os.path.exists("runs/sft-merged/config.json"), "merge did not produce a model"
sh("python train/dpo.py --max-steps 20")

In [ ]:
sh("python train/dpo.py")
assert os.path.exists("runs/dpo/adapter_config.json"), "DPO finished but no adapter was saved"
start("runs/sft-merged", "dpo", adapter="runs/dpo")
sh(
    f"citegate run --agent llm --provider ollama --base-url {URL} --model dpo --concurrency 8 --out runs/dpo.jsonl --html runs/dpo.html --md runs/dpo.md"
)
sh(f"sanad dev-eval --base-url {URL} --model dpo --concurrency 8 --out runs/dpo-dev.jsonl")
sh(f"sanad bench --base-url {URL} --model dpo --n 100 --concurrency 8 --out runs/dpo-bench.json")
stop()
sh("citegate compare runs/base.jsonl runs/sft.jsonl runs/dpo.jsonl --md runs/compare-all.md")

## 6. Keep your results
Download `runs/` (Output tab, or the cell below zips it) and commit the `.jsonl`, `.md`, `.html` and `*-bench.json` files into `results/` in your repo.

In [ ]:
sh("zip -rq sanad-runs.zip runs && ls -la sanad-runs.zip")

In [ ]:
# Optional: publish the adapter to the Hugging Face Hub (free). Needs the HF_TOKEN secret.
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login, HfApi
# login(UserSecretsClient().get_secret("HF_TOKEN"))
# HfApi().create_repo("YOUR_USER/sanad-adapter", exist_ok=True)
# HfApi().upload_folder(folder_path="runs/sft", repo_id="YOUR_USER/sanad-adapter")